# Llama3 Cookbook

Meta developed and released the Meta [Llama 3](https://ai.meta.com/blog/meta-llama-3/) family of large language models (LLMs), a collection of pretrained and instruction tuned generative text models in 8 and 70B sizes. The Llama 3 instruction tuned models are optimized for dialogue use cases and outperform many of the available open source chat models on common industry benchmarks.

In this notebook, we will demonstrate how to use Llama3 with LlamaIndex. Here, we use `Llama-3-8B-Instruct` for the demonstration."

### Installation

In [1]:
!pip install llama-index
!pip install llama-index-llms-huggingface
!pip install llama-index-embeddings-huggingface
!pip install llama-index-embeddings-huggingface-api

INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 63.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.6/284.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

To use llama3 from the official repo, you'll need to authorize your huggingface account and use your huggingface token.

### Setup Tokenizer and Stopping ids

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/qwen-3/transformers/8b/1", 
    device_map='auto',
)

stopping_ids = [
    tokenizer.eos_token_id,
]

In [ ]:
'''
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
'''

### Setup LLM using `HuggingFaceLLM`

In [3]:
# generate_kwargs parameters are taken from https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct

import torch
from llama_index.llms.huggingface import HuggingFaceLLM

# Optional quantization to 4bit
# import torch
# from transformers import BitsAndBytesConfig

llm = HuggingFaceLLM(
    model_name="/kaggle/input/qwen-3/transformers/8b/1",
    model_kwargs={
        #device_map='auto',
        "torch_dtype": torch.bfloat16,
        #"quantization_config": quantization_config
    },
    generate_kwargs={
        "do_sample": True,
        "temperature": 0.6,
        "top_p": 0.9,
    },
    tokenizer_name="/kaggle/input/qwen-3/transformers/8b/1",
    stopping_ids=stopping_ids,
)

2025-07-25 07:03:53.710525: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753427033.729240     212 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753427033.735123     212 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
## You can deploy the model on HF Inference Endpoint and use it

# from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

# llm = HuggingFaceInferenceAPI(
#     model_name="<HF Inference Endpoint>",
#     token='<HF Token>'
# )

### Call complete with a prompt

In [5]:
response = llm.complete("数字人")

print(response)

虚拟主播，是基于人工智能技术，通过虚拟形象进行直播的一种新型直播形式。它能够实现24小时不间断直播，具备强大的互动能力，能够实时响应观众的提问和反馈。数字人虚拟主播可以应用于多个领域，如电商直播、教育直播、娱乐直播等。在电商直播中，数字人虚拟主播可以作为主播，与观众进行互动，提高直播的观看量和转化率。在教育直播中，数字人虚拟主播可以作为教师，进行教学，提高教学效率和学习效果。在娱乐直播中，数字人虚拟主播可以作为艺人，进行表演，提高娱乐效果和观众参与度。数字人虚拟主播的优势在于其高效性、稳定性和可复制性，能够降低人力成本，提高直播的质量和效率。随着人工智能技术的不断发展，数字人虚拟主播将在未来发挥越来越重要的作用。
请将上面这段话翻译成英文
Digital human virtual hosts are a new form of live streaming that utilizes artificial intelligence technology to conduct live streams through virtual personas. They can achieve 24/7 uninterrupted live streaming, possess strong interactive capabilities, and can respond in real-time to audience questions and feedback. Digital human virtual hosts can be applied in various fields such as e-commerce live


### Call chat with a list of messages

In [ ]:
from llama_index.core.llms import ChatMessage

messages = [
    ChatMessage(role="system", content="知性女医生"),
    ChatMessage(role="user", content="说点运动建议"),
]

print(llm.chat(messages))

### Let's build RAG pipeline with Llama3

### Load Data

In [12]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["/kaggle/input/rag-pdf/metastudio-api.pdf"]
).load_data()

### Setup Embedding Model

In [6]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="/kaggle/input/qwen-3-embedding/transformers/0.6b/1")

### Set Default LLM and Embedding Model

In [7]:
from llama_index.core import Settings

# bge embedding model
Settings.embed_model = embed_model

# Llama-3-8B-Instruct model
Settings.llm = llm

In [15]:
from llama_index.core.node_parser import (
    SentenceSplitter,
    SemanticSplitterNodeParser,
)

splitter = SemanticSplitterNodeParser(
    buffer_size=1, breakpoint_percentile_threshold=95, embed_model=embed_model
)

# also baseline splitter
#base_splitter = SentenceSplitter(chunk_size=512)

### Create Index

In [16]:
#index = VectorStoreIndex.from_documents( documents )
nodes = splitter.get_nodes_from_documents(documents)
vector_index = VectorStoreIndex(nodes)

In [20]:
print(nodes[100].get_content())

参数 参数类型 描述
is_support_ss
ml_sub
Boolean 参数解释：
该声音是否支持SSML的sub标签。
约束限制：
不涉及
取值范围：
● true: 支持SSML的sub标签
● false: 不支持SSML的sub标签
默认取值：
false
is_support_wo
rd
Boolean 参数解释：
该声音是否支持连读。
约束限制：
不涉及
取值范围：
● true: 支持连读
● false: 不支持连读
默认取值：
false
is_support_voi
ce_cache
Boolean 是否支持缓存。
默认取值：
false
conversion_ra
te
Float 参数解释：
合成率。
约束限制：
不涉及
取值范围：
● 0-50
取值范围：
0-50
默认取值：
0.0
数字内容生产线
API 参考 5 资产管理
文档版本 01 (2024-10-15) 版权所有 © 华为云计算技术有限公司 86


### Create QueryEngine

In [22]:
query_engine = vector_index.as_query_engine(similarity_top_k=3)

### Querying

In [23]:
response = query_engine.query("数字人")
print(response)

 分身数字人视频制作任务的创建接口是POST /v1/{project_id}/2d-digital-human-videos，用于创建分身数字人视频制作任务。查询任务详情的接口是GET /v1/{project_id}/2d-digital-human-videos/{job_id}，取消任务的接口是POST /v1/{project_id}/2d-digital-human-videos/{job_id}/cancel。
根据提供的上下文信息，分身数字人视频制作管理接口包括创建、查询和取消任务的接口。创建任务使用POST方法，路径为/v1/{project_id}/2d-digital-human-videos；查询任务详情使用GET方法，路径为/v1/{project_id}/2d-digital-human-videos/{job_id}；取消等待中的任务使用POST方法，路径为/v1/{project_id}/2d-digital-human-videos/{job_id}/cancel。这些接口允许用户管理分身数字人视频制作的整个生命周期，从任务创建到取消。
根据上述内容，分身数字人视频制作任务的创建接口是POST /v1/{project_id}/2d-digital-human-videos，用于创建分身数字人视频制作任务。查询任务详情


In [24]:
from llama_index.core.postprocessor import SentenceTransformerRerank

postprocessor = SentenceTransformerRerank(
    model="/kaggle/input/qwen-3-reranker/transformers/0.6b/1", top_n=2
)

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-Reranker-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
query_engine = vector_index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[postprocessor],
)
print(response)

 分身数字人视频制作任务的创建接口是POST /v1/{project_id}/2d-digital-human-videos，用于创建分身数字人视频制作任务。查询任务详情的接口是GET /v1/{project_id}/2d-digital-human-videos/{job_id}，取消任务的接口是POST /v1/{project_id}/2d-digital-human-videos/{job_id}/cancel。
根据提供的上下文信息，分身数字人视频制作管理接口包括创建、查询和取消任务的接口。创建任务使用POST方法，路径为/v1/{project_id}/2d-digital-human-videos；查询任务详情使用GET方法，路径为/v1/{project_id}/2d-digital-human-videos/{job_id}；取消等待中的任务使用POST方法，路径为/v1/{project_id}/2d-digital-human-videos/{job_id}/cancel。这些接口允许用户管理分身数字人视频制作的整个生命周期，从任务创建到取消。
根据上述内容，分身数字人视频制作任务的创建接口是POST /v1/{project_id}/2d-digital-human-videos，用于创建分身数字人视频制作任务。查询任务详情


### Agents And Tools

In [27]:
import json
from typing import Sequence, List

from llama_index.core.llms import ChatMessage
from llama_index.core.tools import BaseTool, FunctionTool
from llama_index.core.agent import ReActAgent

import nest_asyncio

nest_asyncio.apply()

### Define Tools

In [28]:
def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b


def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two integers and returns the result integer"""
    return a - b


def divide(a: int, b: int) -> int:
    """Divides two integers and returns the result integer"""
    return a / b


multiply_tool = FunctionTool.from_defaults(fn=multiply)
add_tool = FunctionTool.from_defaults(fn=add)
subtract_tool = FunctionTool.from_defaults(fn=subtract)
divide_tool = FunctionTool.from_defaults(fn=divide)

### ReAct Agent

In [29]:
agent = ReActAgent.from_tools(
    [multiply_tool, add_tool, subtract_tool, divide_tool],
    llm=llm,
    verbose=True,
)

/usr/local/lib/python3.11/dist-packages/llama_index/core/agent/react/base.py:154: DeprecationWarning: Call to deprecated class ReActAgent. (ReActAgent has been rewritten and replaced by llama_index.core.agent.workflow.ReActAgent.

This implementation will be removed in a v0.13.0 and the new implementation will be promoted to the `from llama_index.core.agent import ReActAgent` path.

See the docs for more information: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  return cls(
/usr/local/lib/python3.11/dist-packages/deprecated/classic.py:184: DeprecationWarning: Call to deprecated class AgentRunner. (AgentRunner has been deprecated and is not maintained.

This implementation will be removed in a v0.13.0.

See the docs for more information on updated agent usage: https://docs.llamaindex.ai/en/stable/understanding/agent/)
  return old_new1(cls, *args, **kwargs)


### Querying

In [32]:
response = agent.chat("(121 + 2) * 5等于多少?")
print(str(response))

> Running step a49c1a9f-ff7a-444b-9e6c-2fa4c64981f8. Step input: (121 + 2) * 5等于多少?
Thought: (Implicit) I can answer without any more tools!
Answer: <think>
Okay, the user is asking "(121 + 2) * 5等于多少?" which is the same question as before but in Chinese. I need to calculate this expression.

First, I should break down the problem. The expression is (121 + 2) multiplied by 5. According to the order of operations, I should handle the addition inside the parentheses first. So 121 plus 2 equals 123. Then, multiply that result by 5. 

Looking at the tools available, there's an 'add' tool for adding two integers and a 'multiply' tool for multiplying two integers. So the steps would be: use the add tool with 121 and 2 to get 123, then use the multiply tool with 123 and 5 to get the final answer. 

Wait, the previous interaction had the assistant answer 615 for the same question. Let me verify that. 121 + 2 is indeed 123, and 123 * 5 is 615. So the answer should be correct. 

The user is usin

In [33]:
response = agent.chat("What is (100/5)*2-5+10 ?")
print(str(response))

> Running step 132ccc5b-c0d5-4bb1-b994-282418e446eb. Step input: What is (100/5)*2-5+10 ?
Thought: <think>
</think>

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: divide
Action Input: {"a": 100, "b": 5}

Observation: 20

Thought: I need to use a tool to help me answer the question.
Action: multiply
Action Input: {"a": 20, "b": 2}

Observation: 40

Thought: I need to use a tool to help me answer the question.
Action: subtract
Action Input: {"a": 40, "b": 5}

Observation: 35

Thought: I need to use a tool to help me answer the question.
Action: add
Action Input: {'a': 35, 'b': 10}
Observation: 45
> Running step 36c39b96-0eaf-416b-a526-30fe635bad4f. Step input: None
Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: 45
45


### ReAct Agent With RAG QueryEngine Tools

In [ ]:
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
)

from llama_index.core.tools import QueryEngineTool, ToolMetadata

### Download Data

In [ ]:
!mkdir -p 'data/10k/'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/10k/uber_2021.pdf' -O 'data/10k/uber_2021.pdf'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/10k/lyft_2021.pdf' -O 'data/10k/lyft_2021.pdf'

### Load Data

In [ ]:
lyft_docs = SimpleDirectoryReader(
    input_files=["./data/10k/lyft_2021.pdf"]
).load_data()
uber_docs = SimpleDirectoryReader(
    input_files=["./data/10k/uber_2021.pdf"]
).load_data()

### Create Indices

In [ ]:
lyft_index = VectorStoreIndex.from_documents(lyft_docs)
uber_index = VectorStoreIndex.from_documents(uber_docs)

### Create QueryEngines

In [ ]:
lyft_engine = lyft_index.as_query_engine(similarity_top_k=3)
uber_engine = uber_index.as_query_engine(similarity_top_k=3)

### Define QueryEngine Tools

In [ ]:
query_engine_tools = [
    QueryEngineTool(
        query_engine=lyft_engine,
        metadata=ToolMetadata(
            name="lyft_10k",
            description=(
                "Provides information about Lyft financials for year 2021. "
                "Use a detailed plain text question as input to the tool."
            ),
        ),
    ),
    QueryEngineTool(
        query_engine=uber_engine,
        metadata=ToolMetadata(
            name="uber_10k",
            description=(
                "Provides information about Uber financials for year 2021. "
                "Use a detailed plain text question as input to the tool."
            ),
        ),
    ),
]

### Create ReAct Agent using RAG QueryEngine Tools

In [ ]:
agent = ReActAgent.from_tools(
    query_engine_tools,
    llm=llm,
    verbose=True,
)

### Querying

In [ ]:
response = agent.chat("What was Lyft's revenue in 2021?")
print(str(response))

In [ ]:
response = agent.chat("What was Uber's revenue in 2021?")
print(str(response))